In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/cleaned/cleaned_superstore.csv",
    encoding="latin1"
)

df["Order Date"] = pd.to_datetime(df["Order Date"])

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully!
Rows: 9994
Columns: 26


In [2]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order ID"].nunique()
total_customers = df["Customer ID"].nunique()

profit_margin = (total_profit / total_sales) * 100
average_order_value = total_sales / total_orders

kpi_summary = {
    "Total Sales": round(total_sales, 2),
    "Total Profit": round(total_profit, 2),
    "Total Orders": total_orders,
    "Total Customers": total_customers,
    "Profit Margin (%)": round(profit_margin, 2),
    "Average Order Value": round(average_order_value, 2)
}

kpi_summary

{'Total Sales': np.float64(2297200.86),
 'Total Profit': np.float64(286397.02),
 'Total Orders': 5009,
 'Total Customers': 793,
 'Profit Margin (%)': np.float64(12.47),
 'Average Order Value': np.float64(458.61)}

In [3]:
category_ai = (
    df.groupby("Category")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Quantity=("Quantity", "sum"),
        Orders=("Order ID", "nunique")
    )
)

category_ai["Profit_Margin_%"] = (
    category_ai["Profit"] /
    category_ai["Sales"] * 100
)

category_ai = category_ai.sort_values(
    "Sales",
    ascending=False
)

display(category_ai)

,Sales,Profit,Quantity,Orders,Profit_Margin_%
Category,,,,,
Technology,836154.0330,145454.9481,6939,1544,17.395712
Furniture,741999.7953,18451.2728,8028,1764,2.486695
Office Supplies,719047.0320,122490.8008,22906,3742,17.035158


In [4]:
region_ai = (
    df.groupby("Region")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Orders=("Order ID", "nunique"),
        Customers=("Customer ID", "nunique")
    )
)

region_ai["Profit_Margin_%"] = (
    region_ai["Profit"] /
    region_ai["Sales"] * 100
)

region_ai = region_ai.sort_values(
    "Sales",
    ascending=False
)

display(region_ai)

,Sales,Profit,Orders,Customers,Profit_Margin_%
Region,,,,,
West,725457.8245,108418.4489,1611,686,14.944831
East,678781.2400,91522.7800,1401,674,13.483399
Central,501239.8908,39706.3625,1175,629,7.921629
South,391721.9050,46749.4303,822,512,11.934342


In [5]:
high_sales_low_margin = category_ai[
    category_ai["Profit_Margin_%"] <
    category_ai["Profit_Margin_%"].median()
]

display(high_sales_low_margin)

,Sales,Profit,Quantity,Orders,Profit_Margin_%
Category,,,,,
Furniture,741999.7953,18451.2728,8028,1764,2.486695


In [6]:
discount_analysis = (
    df.groupby("Discount")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Orders=("Order ID", "nunique")
    )
)

discount_analysis["Profit_Margin_%"] = (
    discount_analysis["Profit"] /
    discount_analysis["Sales"] * 100
)

display(discount_analysis)

,Sales,Profit,Orders,Profit_Margin_%
Discount,,,,
0.00,1.087908e+06,320987.6032,2644,29.505019
0.10,5.436935e+04,9029.1770,89,16.607108
0.15,2.755852e+04,1418.9915,51,5.149012
0.20,7.645944e+05,90337.3060,2407,11.815063
0.30,1.032267e+05,-10369.2774,211,-10.045155
0.32,1.449346e+04,-2391.1377,27,-16.498047
0.40,1.164178e+05,-23057.0504,185,-19.805437
0.45,5.484974e+03,-2493.1111,10,-45.453472
0.50,5.891854e+04,-20506.4281,64,-34.804712


In [7]:
loss_orders = df[df["Profit"] < 0]

print("Loss-making rows:", len(loss_orders))
print(
    "Percentage of rows with loss:",
    round(
        len(loss_orders) / len(df) * 100,
        2
    ),
    "%"
)

display(
    loss_orders[
        [
            "Order ID",
            "Product Name",
            "Category",
            "Sales",
            "Discount",
            "Profit"
        ]
    ].head(20)
)

Loss-making rows: 1871
Percentage of rows with loss: 18.72 %


,Order ID,Product Name,Category,Sales,Discount,Profit
3,US-2015-108966,Bretford CR4500 Series Slim Rectangular Table,Furniture,957.5775,0.45,-383.0310
14,US-2015-118983,Holmes Replacement Filter for HEPA Air Cleaner...,Office Supplies,68.8100,0.80,-123.8580
15,US-2015-118983,Storex DuraTech Recycled Plastic Frosted Binders,Office Supplies,2.5440,0.80,-3.8160
23,US-2017-156909,"Global Deluxe Stacking Chair, Gray",Furniture,71.3720,0.30,-1.0196
27,US-2015-150630,"Riverside Palais Royal Lawyers Bookcase, Royal...",Furniture,3083.4300,0.50,-1665.0522
28,US-2015-150630,Avery Recycled Flexi-View Covers for Binding S...,Office Supplies,9.6180,0.70,-7.0532
32,US-2015-150630,"Acco Pressboard Covers with Storage Hooks, 14 ...",Office Supplies,6.8580,0.70,-5.7150
36,CA-2016-117590,"Electrix Architect's Clamp-On Swing Arm Lamp, ...",Furniture,190.9200,0.60,-147.9630
38,CA-2015-117415,"Atlantic Metals Mobile 3-Shelf Bookcases, Cust...",Furniture,532.3992,0.32,-46.9764
39,CA-2015-117415,"Global Fabric Manager's Chair, Dark Gray",Furniture,212.0580,0.30,-15.1470


In [8]:
loss_products = (
    df.groupby("Product Name")["Profit"]
    .sum()
    .sort_values()
    .head(10)
)

display(loss_products)

Product Name
Cubify CubeX 3D Printer Double Head Print                           -8879.9704
Lexmark MX611dhe Monochrome Laser Printer                           -4589.9730
Cubify CubeX 3D Printer Triple Head Print                           -3839.9904
Chromcraft Bull-Nose Wood Oval Conference Tables & Bases            -2876.1156
Bush Advantage Collection Racetrack Conference Table                -1934.3976
GBC DocuBind P400 Electric Binding System                           -1878.1662
Cisco TelePresence System EX90 Videoconferencing Unit               -1811.0784
Martin Yale Chadless Opener Electric Letter Opener                  -1299.1836
Balt Solid Wood Round Tables                                        -1201.0581
BoxOffice By Design Rectangular and Half-Moon Meeting Room Tables   -1148.4375
Name: Profit, dtype: float64

In [9]:
ai_business_summary = {
    "KPI Summary": kpi_summary,
    "Category Performance": category_ai.round(2).to_dict("index"),
    "Region Performance": region_ai.round(2).to_dict("index"),
    "High Sales Low Margin Categories":
        high_sales_low_margin.round(2).to_dict("index"),
    "Discount Analysis":
        discount_analysis.round(2).to_dict("index"),
    "Top Loss Making Products":
        loss_products.round(2).to_dict()
}

ai_business_summary

{'KPI Summary': {'Total Sales': np.float64(2297200.86),
  'Total Profit': np.float64(286397.02),
  'Total Orders': 5009,
  'Total Customers': 793,
  'Profit Margin (%)': np.float64(12.47),
  'Average Order Value': np.float64(458.61)},
 'Category Performance': {'Technology': {'Sales': 836154.03,
   'Profit': 145454.95,
   'Quantity': 6939,
   'Orders': 1544,
   'Profit_Margin_%': 17.4},
  'Furniture': {'Sales': 741999.8,
   'Profit': 18451.27,
   'Quantity': 8028,
   'Orders': 1764,
   'Profit_Margin_%': 2.49},
  'Office Supplies': {'Sales': 719047.03,
   'Profit': 122490.8,
   'Quantity': 22906,
   'Orders': 3742,
   'Profit_Margin_%': 17.04}},
 'Region Performance': {'West': {'Sales': 725457.82,
   'Profit': 108418.45,
   'Orders': 1611,
   'Customers': 686,
   'Profit_Margin_%': 14.94},
  'East': {'Sales': 678781.24,
   'Profit': 91522.78,
   'Orders': 1401,
   'Customers': 674,
   'Profit_Margin_%': 13.48},
  'Central': {'Sales': 501239.89,
   'Profit': 39706.36,
   'Orders': 1175,


In [10]:
ai_prompt = f"""
You are a senior business data analyst.

Analyze the following e-commerce business data.

BUSINESS KPIs:
{kpi_summary}

CATEGORY PERFORMANCE:
{category_ai.round(2).to_dict("index")}

REGION PERFORMANCE:
{region_ai.round(2).to_dict("index")}

DISCOUNT ANALYSIS:
{discount_analysis.round(2).to_dict("index")}

TOP LOSS-MAKING PRODUCTS:
{loss_products.round(2).to_dict()}

Your task:

1. Identify the most important business trends.
2. Identify high-sales but low-profit areas.
3. Identify potential profitability risks.
4. Analyze the relationship between discount levels and profit.
5. Identify products that may require management attention.
6. Give 5 actionable business recommendations.
7. Clearly separate:
   - Observed facts
   - Possible explanations/hypotheses
   - Recommended actions

Do not claim causation unless the data supports it.
Use specific numbers whenever possible.
"""

print(ai_prompt)


You are a senior business data analyst.

Analyze the following e-commerce business data.

BUSINESS KPIs:
{'Total Sales': np.float64(2297200.86), 'Total Profit': np.float64(286397.02), 'Total Orders': 5009, 'Total Customers': 793, 'Profit Margin (%)': np.float64(12.47), 'Average Order Value': np.float64(458.61)}

CATEGORY PERFORMANCE:
{'Technology': {'Sales': 836154.03, 'Profit': 145454.95, 'Quantity': 6939, 'Orders': 1544, 'Profit_Margin_%': 17.4}, 'Furniture': {'Sales': 741999.8, 'Profit': 18451.27, 'Quantity': 8028, 'Orders': 1764, 'Profit_Margin_%': 2.49}, 'Office Supplies': {'Sales': 719047.03, 'Profit': 122490.8, 'Quantity': 22906, 'Orders': 3742, 'Profit_Margin_%': 17.04}}

REGION PERFORMANCE:
{'West': {'Sales': 725457.82, 'Profit': 108418.45, 'Orders': 1611, 'Customers': 686, 'Profit_Margin_%': 14.94}, 'East': {'Sales': 678781.24, 'Profit': 91522.78, 'Orders': 1401, 'Customers': 674, 'Profit_Margin_%': 13.48}, 'Central': {'Sales': 501239.89, 'Profit': 39706.36, 'Orders': 1175, 

In [11]:
with open(
    "../reports/ai_analysis_prompt.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(ai_prompt)

print("AI analysis prompt saved successfully!")

AI analysis prompt saved successfully!


In [1]:
import pandas as pd
import numpy as np

# Load cleaned data
df = pd.read_csv(
    "../data/cleaned/cleaned_superstore.csv",
    encoding="latin1"
)

# -----------------------------
# 1. BUSINESS KPIs
# -----------------------------

total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order ID"].nunique()
total_customers = df["Customer ID"].nunique()

profit_margin = (
    total_profit / total_sales * 100
    if total_sales != 0 else 0
)

avg_order_value = (
    total_sales / total_orders
    if total_orders != 0 else 0
)

# -----------------------------
# 2. CATEGORY ANALYSIS
# -----------------------------

category_analysis = (
    df.groupby("Category")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Orders=("Order ID", "nunique")
    )
    .reset_index()
)

category_analysis["Profit_Margin"] = (
    category_analysis["Profit"] /
    category_analysis["Sales"] * 100
)

# -----------------------------
# 3. REGION ANALYSIS
# -----------------------------

region_analysis = (
    df.groupby("Region")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Orders=("Order ID", "nunique")
    )
    .reset_index()
)

region_analysis["Profit_Margin"] = (
    region_analysis["Profit"] /
    region_analysis["Sales"] * 100
)

# -----------------------------
# 4. DISCOUNT ANALYSIS
# -----------------------------

discount_analysis = (
    df.groupby("Discount")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Orders=("Order ID", "nunique")
    )
    .reset_index()
)

discount_analysis["Profit_Margin"] = (
    discount_analysis["Profit"] /
    discount_analysis["Sales"] * 100
)

# -----------------------------
# 5. PRODUCT ANALYSIS
# -----------------------------

product_analysis = (
    df.groupby("Product Name")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Quantity=("Quantity", "sum")
    )
    .reset_index()
)

top_products = (
    product_analysis
    .sort_values("Sales", ascending=False)
    .head(10)
)

loss_products = (
    product_analysis[
        product_analysis["Profit"] < 0
    ]
    .sort_values("Profit")
    .head(10)
)

# -----------------------------
# 6. AI BUSINESS CONTEXT
# -----------------------------

best_category = (
    category_analysis
    .sort_values("Profit", ascending=False)
    .iloc[0]
)

worst_category = (
    category_analysis
    .sort_values("Profit")
    .iloc[0]
)

best_region = (
    region_analysis
    .sort_values("Profit", ascending=False)
    .iloc[0]
)

worst_region = (
    region_analysis
    .sort_values("Profit")
    .iloc[0]
)

highest_discount = (
    discount_analysis
    .sort_values("Discount", ascending=False)
    .iloc[0]
)

# -----------------------------
# 7. STRUCTURED AI INPUT
# -----------------------------

ai_context = f"""
E-COMMERCE BUSINESS ANALYSIS

OVERALL KPIs
------------
Total Sales: {total_sales:,.2f}
Total Profit: {total_profit:,.2f}
Total Orders: {total_orders:,}
Total Customers: {total_customers:,}
Profit Margin: {profit_margin:.2f}%
Average Order Value: {avg_order_value:,.2f}

CATEGORY PERFORMANCE
--------------------
{category_analysis.to_string(index=False)}

REGION PERFORMANCE
------------------
{region_analysis.to_string(index=False)}

DISCOUNT PERFORMANCE
--------------------
{discount_analysis.to_string(index=False)}

TOP PRODUCTS BY SALES
---------------------
{top_products.to_string(index=False)}

LOSS-MAKING PRODUCTS
--------------------
{loss_products.to_string(index=False)}

BEST CATEGORY
-------------
{best_category['Category']}
Sales: {best_category['Sales']:,.2f}
Profit: {best_category['Profit']:,.2f}
Margin: {best_category['Profit_Margin']:.2f}%

WORST CATEGORY
--------------
{worst_category['Category']}
Sales: {worst_category['Sales']:,.2f}
Profit: {worst_category['Profit']:,.2f}
Margin: {worst_category['Profit_Margin']:.2f}%

BEST REGION
-----------
{best_region['Region']}
Sales: {best_region['Sales']:,.2f}
Profit: {best_region['Profit']:,.2f}

WORST REGION
------------
{worst_region['Region']}
Sales: {worst_region['Sales']:,.2f}
Profit: {worst_region['Profit']:,.2f}
"""

print(ai_context)


E-COMMERCE BUSINESS ANALYSIS

OVERALL KPIs
------------
Total Sales: 2,297,200.86
Total Profit: 286,397.02
Total Orders: 5,009
Total Customers: 793
Profit Margin: 12.47%
Average Order Value: 458.61

CATEGORY PERFORMANCE
--------------------
       Category       Sales      Profit  Orders  Profit_Margin
      Furniture 741999.7953  18451.2728    1764       2.486695
Office Supplies 719047.0320 122490.8008    3742      17.035158
     Technology 836154.0330 145454.9481    1544      17.395712

REGION PERFORMANCE
------------------
 Region       Sales      Profit  Orders  Profit_Margin
Central 501239.8908  39706.3625    1175       7.921629
   East 678781.2400  91522.7800    1401      13.483399
  South 391721.9050  46749.4303     822      11.934342
   West 725457.8245 108418.4489    1611      14.944831

DISCOUNT PERFORMANCE
--------------------
 Discount        Sales      Profit  Orders  Profit_Margin
     0.00 1087908.4700 320987.6032    2644      29.505019
     0.10   54369.3510   9029.177

In [2]:
ai_prompt = f"""
You are a senior business analyst reviewing an e-commerce dataset.

Analyze the following verified business metrics:

{ai_context}

Generate an executive business analysis with these sections:

1. Executive Summary
2. Key Business Insights
3. Profitability Risks
4. Sales Opportunities
5. Customer/Product Insights
6. Discount Strategy
7. Regional Opportunities
8. Five Actionable Recommendations

IMPORTANT RULES:

- Only use facts supported by the provided data.
- Do not invent numbers.
- Clearly distinguish observed facts from hypotheses.
- Do not claim that discount causes lower profit unless the data proves causation.
- Focus on actionable business decisions.
- Keep the recommendations practical.
"""

print(ai_prompt)


You are a senior business analyst reviewing an e-commerce dataset.

Analyze the following verified business metrics:


E-COMMERCE BUSINESS ANALYSIS

OVERALL KPIs
------------
Total Sales: 2,297,200.86
Total Profit: 286,397.02
Total Orders: 5,009
Total Customers: 793
Profit Margin: 12.47%
Average Order Value: 458.61

CATEGORY PERFORMANCE
--------------------
       Category       Sales      Profit  Orders  Profit_Margin
      Furniture 741999.7953  18451.2728    1764       2.486695
Office Supplies 719047.0320 122490.8008    3742      17.035158
     Technology 836154.0330 145454.9481    1544      17.395712

REGION PERFORMANCE
------------------
 Region       Sales      Profit  Orders  Profit_Margin
Central 501239.8908  39706.3625    1175       7.921629
   East 678781.2400  91522.7800    1401      13.483399
  South 391721.9050  46749.4303     822      11.934342
   West 725457.8245 108418.4489    1611      14.944831

DISCOUNT PERFORMANCE
--------------------
 Discount        Sales      Pr

In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found. Check your .env file."
    )

client = OpenAI(api_key=api_key)

response = client.responses.create(
    model="gpt-5-mini",
    input=ai_prompt
)

ai_output = response.output_text

print(ai_output)

ValueError: OPENAI_API_KEY not found. Check your .env file.

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

print("API key found:", api_key is not None)

API key found: True


In [5]:
from pathlib import Path
import os

print("Notebook working directory:")
print(Path.cwd())

print("\n.env exists:")
print(Path(".env").exists())

print("\nProject root .env exists:")
print(Path("../.env").exists())

Notebook working directory:
c:\Users\piyus\Desktop\Data Analysts  projects\AI-Powered E-Commerce Data Analyst\notebooks

.env exists:
False

Project root .env exists:
True


In [6]:
from dotenv import load_dotenv
import os

# Load .env from project root
load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

print("API key found:", api_key is not None)

API key found: True


In [10]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

gemini_key = os.getenv("GEMINI_API_KEY")

print("Gemini API key found:", gemini_key is not None)

Gemini API key found: False


In [12]:
from pathlib import Path

env_path = Path("../.env")

print("ENV EXISTS:", env_path.exists())
print("ENV PATH:", env_path.resolve())

if env_path.exists():
    print("\nENV CONTENT (key hidden):")

    for line in env_path.read_text(encoding="utf-8").splitlines():
        if "GEMINI_API_KEY" in line:
            key_part = line.split("=", 1)[1].strip() if "=" in line else ""
            print("GEMINI_API_KEY =", key_part[:8] + "..." if key_part else "EMPTY")
        elif line.strip():
            print(line.split("=")[0] + "=...")



ENV EXISTS: True
ENV PATH: C:\Users\piyus\Desktop\Data Analysts  projects\AI-Powered E-Commerce Data Analyst\.env

ENV CONTENT (key hidden):
OPENAI_API_KEY=...


In [15]:
from dotenv import load_dotenv
import os

load_dotenv("../.env", override=True)

gemini_key = os.getenv("GEMINI_API_KEY")

print("Gemini API key found:", gemini_key is not None)

Gemini API key found: False
